#---Importing Libraries---

In [14]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

# -- Step 1: Loading and Exploring the Dataset --

In [15]:
df= pd.read_csv('newss.csv')

print("first few rowa")
print(df.head())
print("\nFake vs Real counts:")
print(df['label'].value_counts())

first few rowa
                                                text  label
0  Health ministry expands vaccination program na...      0
1  Researchers find evidence of dragons living in...      1
2  Education ministry announces scholarship oppor...      0
3  New environmental law aims to reduce industria...      0
4  Secret lab develops invisible soldiers for fut...      1

Fake vs Real counts:
label
0    60
1    40
Name: count, dtype: int64


# --- Step 2: Text Preprocessing ---
# Using TF-IDF Vectorization to convert text to numerical features

In [16]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text']).toarray()
y = df['label'].values

# --- Step 3: Spliting the Dataset ---
# 80% Training, 20% Testing

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Converting to PyTorch Tensors

In [18]:
X_train_t = torch.tensor(X_train).float()
y_train_t = torch.tensor(y_train).float().view(-1, 1)
X_test_t = torch.tensor(X_test).float()
y_test_t = torch.tensor(y_test).float().view(-1, 1)

# --- Step 4: Building a Neural Network ---

In [19]:
class NeuralNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # Architecture: Input -> 128 -> 64 -> Output
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),                  # ReLU for hidden layers
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()                # Sigmoid for output layer
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train.shape[1]
model = NeuralNet(input_dim)

# --- Step 5: Training the Model ---

In [20]:
epochs = 20
learning_rate = 0.001
criterion = nn.BCELoss() # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [5/20], Loss: 0.6775
Epoch [10/20], Loss: 0.6612
Epoch [15/20], Loss: 0.6370
Epoch [20/20], Loss: 0.6011


# --- Step 6: Evaluating the Model ---

In [21]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_t)
    # Convert probabilities to 0 or 1
    predicted_classes = (predictions > 0.5).float()
    acc = (predicted_classes == y_test_t).sum().item() / y_test_t.size(0)
    print(f"\nFinal Accuracy: {acc * 100:.2f}%")



Final Accuracy: 75.00%


# --- Step 7: Testing Model ---

In [22]:
def manual_test(headline):
    model.eval()
    with torch.no_grad():
        vec = vectorizer.transform([headline]).toarray()
        vec_t = torch.tensor(vec).float()
        prob = model(vec_t).item()
        result = "Fake News" if prob > 0.5 else "Real News"
        print(f"Headline: {headline} -> Prediction: {result}")

manual_test("Scientists confirm water on Mars.")
manual_test("Secret government project creates invisible humans.")

Headline: Scientists confirm water on Mars. -> Prediction: Real News
Headline: Secret government project creates invisible humans. -> Prediction: Real News
